# 05 — Hand Keypoints: Draw the Skeleton, Understand the 21 Numbers

## 1. Learning Goal

You will be able to explain, **and demonstrate**:

> A hand pose is just 21 points. 21 points = 42 numbers a classifier can use.

1. What **keypoints / landmarks** are, and how they differ from a bounding box
2. Convert model output (fractions) into **pixels** and draw the **skeleton**
3. Read the 21 landmarks as a **table of numbers**
4. Why raw coordinates are a problem, and how **normalization** fixes it
5. Run it **live on your webcam** with FPS (Milestone 2)
6. *(Optional)* Train **your own** keypoint model with Ultralytics YOLO-pose and compare it with MediaPipe

## 2. Concept

```text
 Box:        "a hand is in this rectangle"          → 4 numbers
 Keypoints:  "the thumb tip is here, the wrist..."   → 21 × (x, y) = 42 numbers
```

```text
Landmark numbering (the same on every hand)

            8    12   16   20
            |    |    |    |
            7    11   15   19        4
            |    |    |    |         |
            6    10   14   18        3
            |    |    |    |        /
            5----9----13---17      2
             \                /   /
              \______________/  1
                     0   (wrist)
```

## 3. Why do we need this?

A CNN on raw pixels has to learn what a hand is, in every light and background. Keypoints throw the background away and keep only the **shape of the hand**. That makes a small, simple classifier possible (notebook 06).

In [ ]:
import time
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Works whether you start Jupyter in the project folder or inside notebooks/
ROOT = Path("..") if Path("../data").exists() else Path(".")
DATA = ROOT / "data"
MODELS = ROOT / "models"
DATA.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

def show_bgr(img, title="", size=6):
    plt.figure(figsize=(size, size))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))   # OpenCV is BGR, Matplotlib wants RGB
    plt.title(title)
    plt.axis("off")
    plt.show()

def describe(image, name="image"):
    print(f"--- {name} ---")
    print("shape :", image.shape, "| dtype:", image.dtype, "| min:", image.min(), "| max:", image.max())

In [ ]:
import mediapipe as mp

def create_detector(running_mode="image", num_hands=1, min_detection=0.5):
    """Load the MediaPipe Hand Landmarker model file from models/."""
    modes = {"image": mp.tasks.vision.RunningMode.IMAGE,
             "video": mp.tasks.vision.RunningMode.VIDEO}
    options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODELS / "hand_landmarker.task")),
        running_mode=modes[running_mode],
        num_hands=num_hands,
        min_hand_detection_confidence=min_detection,
    )
    return mp.tasks.vision.HandLandmarker.create_from_options(options)

def detect_hand(detector, bgr_image):
    """OpenCV gives BGR, MediaPipe wants RGB. Convert, wrap, detect."""
    rgb = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    return detector.detect(mp_image)

def landmarks_to_array(result, hand_index=0):
    """result -> NumPy array of shape (21, 3): columns are x, y, z (x and y are fractions 0..1)."""
    hand = result.hand_landmarks[hand_index]
    return np.array([[lm.x, lm.y, lm.z] for lm in hand], dtype=np.float32)

In [ ]:
# Which landmark connects to which (pairs of landmark numbers)
CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),          # thumb
    (0, 5), (5, 6), (6, 7), (7, 8),          # index finger
    (5, 9), (9, 10), (10, 11), (11, 12),     # middle finger
    (9, 13), (13, 14), (14, 15), (15, 16),   # ring finger
    (13, 17), (17, 18), (18, 19), (19, 20),  # little finger
    (0, 17),                                  # outer edge of the palm
]

LANDMARK_NAMES = [
    "WRIST",
    "THUMB_CMC", "THUMB_MCP", "THUMB_IP", "THUMB_TIP",
    "INDEX_MCP", "INDEX_PIP", "INDEX_DIP", "INDEX_TIP",
    "MIDDLE_MCP", "MIDDLE_PIP", "MIDDLE_DIP", "MIDDLE_TIP",
    "RING_MCP", "RING_PIP", "RING_DIP", "RING_TIP",
    "PINKY_MCP", "PINKY_PIP", "PINKY_DIP", "PINKY_TIP",
]

---
# Part 1 — Draw the skeleton (Phase 1, Step 3)

## 4. Visual intuition

The model gives **fractions** (0..1). To draw, convert to **pixels**:

```text
pixel_x = x * width         pixel_y = y * height
```

Careful: OpenCV drawing functions want `(x, y)`, but NumPy images are indexed `image[row, col]` = `image[y, x]`.

In [ ]:
PHOTO = DATA / "myhand.jpg"
img = cv2.imread(str(PHOTO))
H, W = img.shape[:2]

detector = create_detector()
result = detect_hand(detector, img)
lm = landmarks_to_array(result)                           # (21, 3) fractions

points_px = (lm[:, :2] * np.array([W, H])).astype(int)    # (21, 2) pixels, columns (x, y)
print("hands:", len(result.hand_landmarks), "| points_px shape:", points_px.shape)

In [ ]:
def pt(p):
    """cv2 wants plain Python ints in a tuple, not NumPy integers."""
    return int(p[0]), int(p[1])

def draw_skeleton(bgr, points_px, show_numbers=False, line_colour=(0, 255, 0), dot_colour=(0, 0, 255)):
    out = bgr.copy()
    for a, b in CONNECTIONS:                                   # lines first...
        cv2.line(out, pt(points_px[a]), pt(points_px[b]), line_colour, 2)
    for i, p in enumerate(points_px):                          # ...dots on top
        cv2.circle(out, pt(p), 5, dot_colour, -1)
        if show_numbers:
            cv2.putText(out, str(i), (int(p[0]) + 6, int(p[1]) - 6),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1)
    return out

show_bgr(draw_skeleton(img, points_px), "Skeleton")
show_bgr(draw_skeleton(img, points_px, show_numbers=True), "Skeleton with landmark numbers")

### What to observe

- 21 dots: one on the wrist (0), four on each finger.
- The numbers go **wrist → base of finger → tip**. Fingertips are 4, 8, 12, 16, 20.

### Challenge 1

1. Colour the **thumb** differently from the other fingers (hint: loop over the first four connections separately).
2. What happens if you write `lm[:, 0] * H` (swap width/height)? Predict, then try it on a non-square photo.

---
# Part 2 — The 21 numbers as a table (Step 3, continued)

In [ ]:
import pandas as pd

table = pd.DataFrame({
    "name": LANDMARK_NAMES,
    "x_frac": lm[:, 0].round(3),
    "y_frac": lm[:, 1].round(3),
    "z": lm[:, 2].round(3),
    "x_px": points_px[:, 0],
    "y_px": points_px[:, 1],
})
table

In [ ]:
plt.figure(figsize=(5, 6))
for a, b in CONNECTIONS:
    plt.plot([points_px[a, 0], points_px[b, 0]], [points_px[a, 1], points_px[b, 1]], "g-")
plt.scatter(points_px[:, 0], points_px[:, 1], c="red")
for i, (x, y) in enumerate(points_px):
    plt.text(x + 4, y - 4, str(i), fontsize=8)
plt.gca().invert_yaxis()                 # image coordinates: y grows DOWNWARD
plt.gca().set_aspect("equal")
plt.title("The same hand, as pure numbers (no photo!)")
plt.show()

### What to observe

- The table has everything a classifier needs: shape information without any background.
- In the plot we flipped the y axis. Without `invert_yaxis()` your hand appears upside down, because in images `y = 0` is the **top**.
- `z` is relative depth with the wrist as origin. We will mostly use `x, y` first.

---
# Part 3 — Why raw coordinates are not enough: normalization

## Concept

Imagine the **same hand shape** shown (a) near the camera, (b) far away and in a different corner. The raw numbers are totally different, but the sign is the same!

```text
we want:   same shape  →  same features
```

Fix in two steps:

```text
1. Translate: subtract the wrist, so the wrist is always (0, 0)
2. Scale:     divide by the hand size, so a big hand and a small hand look alike
```

In [ ]:
def normalize_keypoints(xy):
    """xy: array (21, 2) of pixel coordinates -> same shape, wrist at (0, 0), hand size = 1."""
    xy = np.asarray(xy, dtype=np.float32)
    centred = xy - xy[0]                           # 1. translate: wrist becomes the origin
    scale = np.linalg.norm(centred[9])             # 2. hand size = distance wrist -> middle finger base
    if scale < 1e-6:
        scale = 1.0
    return centred / scale

xy = points_px.astype(np.float32)

# Fake a second view of the SAME hand: half the size, moved to another place
xy_far = xy * 0.5 + np.array([200, 100], dtype=np.float32)

raw_difference = np.abs(xy - xy_far).mean()
norm_difference = np.abs(normalize_keypoints(xy) - normalize_keypoints(xy_far)).mean()
print(f"mean difference, raw        : {raw_difference:8.2f}")
print(f"mean difference, normalized : {norm_difference:8.5f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (a, b), title in zip(axes,
        [(xy, xy_far), (normalize_keypoints(xy), normalize_keypoints(xy_far))],
        ["RAW: two views look very different", "NORMALIZED: the same shape"]):
    ax.scatter(a[:, 0], a[:, 1], c="tab:blue", label="view 1")
    ax.scatter(b[:, 0], b[:, 1], c="tab:orange", label="view 2 (smaller, moved)")
    ax.invert_yaxis(); ax.set_aspect("equal"); ax.legend(); ax.set_title(title)
plt.show()

### What to observe

- Raw difference is large; normalized difference is ~0 (the two orange/blue clouds overlap exactly).
- This is the key idea: **we remove information that does not matter** (where the hand is, how big it looks).

### What normalization does NOT fix

**Rotation** (tilting the hand) and **mirroring** (left vs right hand). Think about whether a sign should stay the same when rotated.

### Challenge 2

1. Rotate `xy` by 30° around the wrist and compare normalized difference. What do you see?
2. Why do we divide by distance wrist → landmark 9, and not by the image width?

---
# Part 4 — From keypoints to a feature vector

A classifier wants **one row of numbers per example**:

```text
21 points × (x, y) = 42 numbers, but the wrist is always (0, 0), so we drop it → 40 features
```

In [ ]:
def to_features(xy):
    """(21, 2) pixel keypoints -> 1D vector of 40 numbers (normalized, wrist removed)."""
    return normalize_keypoints(xy)[1:].flatten()

features = to_features(xy)
print("feature vector shape:", features.shape)
print(features.round(2))

This vector is the input to notebook 06. **Photo of a hand → 40 numbers → sign.**

---
# Part 5 — Live webcam: skeleton + FPS (Milestone 2)  [Step 6]

## Concept

```text
read frame ─► mirror ─► detect ─► draw skeleton ─► show ─► repeat
```

Video mode needs a **timestamp in milliseconds that always increases**. FPS is `1 / time taken for one loop`.

Set `RUN_LIVE = True`, run, and press **q** in the video window to stop. This needs a desktop with a webcam (it will not work on a remote server).

> On macOS, if the window freezes, run it from a normal Python script instead of the notebook.

In [ ]:
RUN_LIVE = False

if RUN_LIVE:
    live_detector = create_detector("video")
    cap = cv2.VideoCapture(0)
    start = time.perf_counter()
    last_ts, prev, fps = -1, start, 0.0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)                              # selfie view

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        ts = max(int((time.perf_counter() - start) * 1000), last_ts + 1)   # must strictly increase
        last_ts = ts
        res = live_detector.detect_for_video(mp_image, ts)

        if res.hand_landmarks:
            h, w = frame.shape[:2]
            pts = (landmarks_to_array(res)[:, :2] * np.array([w, h])).astype(int)
            frame = draw_skeleton(frame, pts)

        now = time.perf_counter()
        fps = 0.9 * fps + 0.1 * (1.0 / max(now - prev, 1e-6))   # smoothed FPS
        prev = now
        cv2.putText(frame, f"FPS: {fps:.1f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.imshow("skeleton (q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

### What to observe

- The skeleton follows your hand. Move it fast, tilt it, cover half of it. When does it fail?
- Note the FPS number. Close other programs and compare.
- We **mirror** the frame (`cv2.flip`). From now on, **all training data and all live inference must use the same mirroring**, or the model sees mirrored hands it never trained on. Handedness labels (Left/Right) also swap with the flip, so we will not rely on them.

### Challenge 3

1. Try `num_hands=2`. What changes in `res.hand_landmarks`?
2. Show the **x pixel position of the index fingertip** on screen as text.
3. Draw the box from notebook 04 around the hand live.

---
# Part 6 (optional) — Train your OWN keypoint model with YOLO-pose  [Step 5]

## Concept

MediaPipe is **pretrained**: someone else trained it. Here you do the training yourself with the Ultralytics hand-keypoints dataset (about 26,000 images with 21 keypoints each; non-commercial licence; its labels were made by MediaPipe).

```text
images + keypoint labels ─► YOLO-pose training ─► your model ─► skeleton
```

You will see **training and validation loss curves**, a preview of notebook 08.

**Requirements:** `pip install ultralytics`. The first run downloads the dataset (~370 MB). Training is slow on a CPU. A free GPU on Google Colab or Kaggle is recommended. Start with `epochs=5` just to see it work.

> The pretrained starting weights are named like `yolo11n-pose.pt`. If your Ultralytics docs list a newer name, use that.

In [ ]:
RUN_YOLO = False

if RUN_YOLO:
    from ultralytics import YOLO

    yolo = YOLO("yolo11n-pose.pt")                     # small pretrained pose model (starting point)
    train_results = yolo.train(
        data="hand-keypoints.yaml",                    # Ultralytics downloads this dataset automatically
        epochs=5,                                      # EXP-002: try 5, 20, 50
        imgsz=640,
        batch=16,
        project=str(ROOT / "experiments"),
        name="hand_pose_exp002",
    )

In [ ]:
# Loss curves from the training run (works after the training above has finished)
import pandas as pd

results_csv = ROOT / "experiments" / "hand_pose_exp002" / "results.csv"
if results_csv.exists():
    log = pd.read_csv(results_csv)
    log.columns = log.columns.str.strip()
    print(list(log.columns))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, name in zip(axes, ["pose_loss", "box_loss"]):
        ax.plot(log["epoch"], log[f"train/{name}"], label="train")
        ax.plot(log["epoch"], log[f"val/{name}"], label="validation")
        ax.set_title(name); ax.set_xlabel("epoch"); ax.legend(); ax.grid(True)
    plt.show()
else:
    print("No training log yet. Set RUN_YOLO = True first (or skip this optional part).")

In [ ]:
# Compare YOUR model with MediaPipe on the same photo
best = ROOT / "experiments" / "hand_pose_exp002" / "weights" / "best.pt"
if best.exists():
    from ultralytics import YOLO
    my_model = YOLO(str(best))
    r = my_model.predict(str(PHOTO), verbose=False)[0]

    if r.keypoints is not None and len(r.keypoints.xy) > 0:
        yolo_xy = r.keypoints.xy[0].cpu().numpy()          # (21, 2) pixels
        hand_size = np.linalg.norm(points_px[9] - points_px[0])
        error = np.linalg.norm(yolo_xy - points_px, axis=1).mean() / hand_size
        print(f"average difference vs MediaPipe: {error:.1%} of the hand size")

        both = draw_skeleton(img, points_px, line_colour=(0, 255, 0), dot_colour=(0, 255, 0))
        both = draw_skeleton(both, yolo_xy.astype(int), line_colour=(0, 0, 255), dot_colour=(0, 0, 255))
        show_bgr(both, "green = MediaPipe, red = your YOLO-pose")
    else:
        print("Your model found no hand in this photo.")
else:
    print("No trained weights found yet.")

### What to observe / think about

- Does the validation loss follow the training loss, or does it go flat or up?
- After 5 epochs vs 20 epochs, how different are the skeletons?
- Because the labels came from MediaPipe, your model can at best **copy MediaPipe**, never beat it. Why? What does that say about the quality of labels?

---
# 9. Results

```text
photo ─► 21 landmarks (fractions) ─► pixels ─► skeleton drawing
                      │
                      └─► normalize (wrist = origin, size = 1) ─► 40 features ─► (notebook 06)
```

# 10. What did we learn?

1. Keypoints describe hand **shape** with 21 `(x, y)` points, independent of the background.
2. Fractions → pixels: `x * width`, `y * height`.
3. Plotting keypoints needs `invert_yaxis()`; image y points down.
4. **Normalization** (translate + scale) makes the same shape give the same numbers.
5. Live use = a loop: read, mirror, detect, draw, FPS.
6. A model is only as good as its labels.

# 11. Questions to think about

1. Which landmarks would differ most between the signs "A" (fist) and "B" (flat hand)?
2. Why do we mirror the webcam image, and why must training use the same mirroring?
3. Normalization ignores rotation. For which signs is that a problem?
4. What information does a keypoint representation **lose** compared with the full image? (Think about facial expression, hand colour, objects.)

# 12. Mini challenge

Write `finger_is_extended(xy, finger)` that returns `True` when the fingertip is farther from the wrist than the finger's middle joint (e.g. tip 8 vs joint 6 for the index finger). Test it on your own fist and open hand photos. This is a tiny "hand-written classifier" and sets the scene for notebook 06.

---
## Experiment log

```text
EXP-002
Notebook : 05_hand_keypoints
Model    : YOLO-pose (yolo11n-pose) on hand-keypoints
Epochs   : 5 / 20 / 50
Observed : (loss curves, difference vs MediaPipe)
Conclusion :
```

**Phase 1 complete when:** you have a skeleton on a photo, a live skeleton with FPS on the webcam, and you can explain the 21 numbers.
**Next:** `03_dataset_exploration.ipynb`, then `06_baseline_classifier.ipynb`.